In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
ROOT_DIR = Path('/projects/ai_science_alphafold')
RAW_DIR = ROOT_DIR / 'raw'
OUT_DIR = ROOT_DIR / 'processed'

run_config = pd.read_json(RAW_DIR / 'run_config.json').iloc[0]
MATCHING_FIRST_YEAR = int(run_config['MATCHING_FIRST_YEAR'])

candidate_scientists = pd.read_csv(OUT_DIR / 'candidate_scientists.csv')
pub2year = pd.read_csv(OUT_DIR / 'pub2year.csv')
pub2aff_main = pd.read_csv(OUT_DIR / 'pub2aff_main.csv')
pub2aff = pd.read_csv(OUT_DIR / 'pub2aff.csv')
pub2ids = pd.read_csv(OUT_DIR / 'pub2ids.csv')

In [ ]:
source2issn = pd.read_csv(
    RAW_DIR / 'source2rank.csv', header=None, names=['source_id', 'issn_l', 'issn'], dtype='string'
)
source2issn['source_id'] = source2issn['source_id'].str.replace('https://openalex.org/', '', regex=False)

issn_long = source2issn.assign(issn=source2issn['issn'].fillna('').str.split(';')).explode('issn')
issn_long = (
    pd.concat(
        [
            issn_long[['source_id', 'issn_l']].rename(columns={'issn_l': 'value'}),
            issn_long[['source_id', 'issn']].rename(columns={'issn': 'value'}),
        ],
        ignore_index=True,
    )
    .dropna()
    .drop_duplicates()
)

years = pd.DataFrame({'Year': np.arange(1997, 2025, dtype='int64')})
source2issn = issn_long.merge(years, how='cross')

issn2rank = pd.read_csv(RAW_DIR / 'journal_information.csv')
issn2rank = issn2rank[['jcrYear', 'jif2019', 'issn', 'eissn', 'quartile']].copy()
issn2rank = issn2rank.loc[issn2rank['quartile'].ne('N/A')].copy()
issn2rank = pd.concat(
    [
        issn2rank[['jcrYear', 'quartile', 'jif2019', 'issn']].rename(columns={'issn': 'value'}),
        issn2rank[['jcrYear', 'quartile', 'jif2019', 'eissn']].rename(columns={'eissn': 'value'}),
    ],
    ignore_index=True,
).dropna(subset=['value'])
issn2rank['Year'] = issn2rank['jcrYear'].astype('int64') + 1
issn2rank['jif'] = pd.to_numeric(issn2rank['jif2019'].replace({'<0.1': np.nan, 'N/A': np.nan}), errors='coerce').fillna(
    0.0
)
issn2rank = issn2rank[['Year', 'value', 'quartile', 'jif']]

source2rank = source2issn.merge(issn2rank, on=['Year', 'value'], how='left')
source2rank = source2rank.loc[source2rank['quartile'].notna()].copy()
source2rank = source2rank.sort_values(['source_id', 'Year', 'quartile']).drop_duplicates(
    ['source_id', 'Year'], keep='first'
)
source2rank = source2rank.rename(columns={'source_id': 'JournalID'})

print(source2rank.shape)

In [ ]:
alphafold_1 = pd.read_csv(RAW_DIR / 'cite_alphafold.csv')
alphafold_2 = pd.read_csv(RAW_DIR / 'mention_alphafold_title.csv')
alphafold_3 = pd.read_csv(
    RAW_DIR / 'mention_alphafold_title.csv',
    header=None,
    names=['PubID', 'mention_alphafold_in_abstract'],
    dtype={'PubID': 'string'},
)

alphafold = alphafold_1.merge(alphafold_2, on='PubID', how='outer').merge(alphafold_3, on='PubID', how='outer')
alphafold['mention_alphafold_in_abstract'] = (
    pd.to_numeric(alphafold['mention_alphafold_in_abstract'], errors='coerce').fillna(0) > 0
)
for c in [x for x in alphafold.columns if x != 'PubID']:
    alphafold[c] = alphafold[c].fillna(False).astype(bool)
alphafold['related_alphafold'] = alphafold[[x for x in alphafold.columns if x != 'PubID']].any(axis=1)

print(alphafold.shape)

In [ ]:
active_authors = set(candidate_scientists['author_id'])
year_product = pub2aff_main.loc[pub2aff_main['author_id'].isin(active_authors)].copy()
year_product['magid'] = year_product['oaid'].astype(str).str.lstrip('W').astype('int64')
year_product = year_product.merge(
    pub2year.loc[pub2year['Year'].between(MATCHING_FIRST_YEAR, 2024)],
    left_on='magid',
    right_on='PaperID',
    how='inner',
)

flagship = {'S137773608', 'S3880285', 'S110447773', 'S125754415'}
year_product['flagship_journal'] = year_product['JournalID'].isin(flagship)

cite_set = set(alphafold.loc[alphafold['cite_alphafold'], 'PubID']) if 'cite_alphafold' in alphafold.columns else set()
mention_set = set(alphafold.loc[alphafold['mention_alphafold_in_abstract'], 'PubID'])
year_product['cite_alphafold'] = year_product['oaid'].isin(cite_set)
year_product['mention_alphafold_in_abstract'] = year_product['oaid'].isin(mention_set)

year_product = year_product.merge(
    source2rank[['JournalID', 'Year', 'quartile', 'jif']], on=['JournalID', 'Year'], how='left'
)
for q in ['Q1', 'Q2', 'Q3', 'Q4']:
    year_product[f'quartile_{q.lower()}'] = year_product['quartile'].eq(q)
year_product['quartile_notone'] = ~year_product['quartile'].eq('Q1')
year_product['quartile_onetwo'] = year_product['quartile'].isin(['Q1', 'Q2'])
year_product['quartile_threefour'] = year_product['quartile'].isin(['Q3', 'Q4'])
year_product['jif'] = pd.to_numeric(year_product['jif'], errors='coerce').fillna(0.0)

pub2unnested = pd.read_csv(RAW_DIR / 'pub2unnested.csv')[
    ['id', 'countries_distinct_count', 'institutions_distinct_count']
]
year_product = year_product.merge(pub2unnested, left_on='oaid', right_on='id', how='left')

agg = (
    year_product.groupby(['author_id', 'Year'], dropna=False)
    .agg(
        pub_count=('oaid', 'size'),
        flagship_journal=('flagship_journal', 'sum'),
        quartile_one=('quartile_q1', 'sum'),
        quartile_two=('quartile_q2', 'sum'),
        quartile_three=('quartile_q3', 'sum'),
        quartile_four=('quartile_q4', 'sum'),
        quartile_notone=('quartile_notone', 'sum'),
        quartile_onetwo=('quartile_onetwo', 'sum'),
        quartile_threefour=('quartile_threefour', 'sum'),
        team_size=('author_order', 'sum'),
        countries_distinct_count=('countries_distinct_count', 'mean'),
        institutions_distinct_count=('institutions_distinct_count', 'mean'),
        cite_mean=('cite_alphafold', 'mean'),
        mention_mean=('mention_alphafold_in_abstract', 'mean'),
        cite_sum=('cite_alphafold', 'sum'),
        mention_sum=('mention_alphafold_in_abstract', 'sum'),
        jif=('jif', 'sum'),
    )
    .reset_index()
)
agg['team_size'] = agg['team_size'] - agg['pub_count']

wide = agg.pivot(index='author_id', columns='Year')
wide.columns = [f'{a}_Year_{b}' for a, b in wide.columns]
year_product_wide = wide.reset_index().fillna(0)

print(year_product_wide.shape)

In [ ]:
def first_mode(x: pd.Series) -> object:
    x = x.dropna()
    if x.empty:
        return pd.NA
    m = x.mode()
    return m.sort_values().iloc[0] if not m.empty else pd.NA


def nunique_exploded_lists(series: pd.Series) -> int:
    vals = []
    for x in series:
        if isinstance(x, list):
            vals.extend(x)
    return pd.Series(vals).dropna().nunique() if vals else 0


author_stats = (
    candidate_scientists.groupby('author_id', dropna=False)
    .agg(
        is_top_aff=('is_top_aff', 'any'),
        from_bio_med=('from_bio_med', 'first'),
        FieldID=('FieldID', first_mode),
        n_affs=('affs_id', nunique_exploded_lists),
        n_tops=('FieldID', nunique_exploded_lists),
        pub_count=('PaperID', 'size'),
        pdb_percent=('is_pdb_paper', 'mean'),
        mean_team_size=('author_order', 'mean'),
    )
    .reset_index()
)
author_stats['treated'] = author_stats['pdb_percent'] > 0
author_stats = author_stats.merge(year_product_wide, on='author_id', how='left').fillna(0)

author_stats.to_csv(OUT_DIR / 'author_stats.csv', index=False)
year_product_wide.to_csv(OUT_DIR / 'year_product_wide.csv', index=False)

print(author_stats.shape)

In [ ]:
author_career_1 = pub2aff_main[['oaid', 'author_id']].copy()
author_career_1 = author_career_1.loc[author_career_1['author_id'].isin(set(author_stats['author_id']))]
author_career_1 = author_career_1.merge(pub2ids[['oaid', 'magid']], on='oaid', how='left')
author_career_1 = author_career_1.merge(pub2year[['PaperID', 'Year']], left_on='magid', right_on='PaperID', how='inner')
author_career_1 = (
    author_career_1.groupby('author_id', as_index=False)['Year']
    .min()
    .rename(columns={'Year': 'first_independent_year'})
)

author_career_2 = pub2aff[['oaid', 'author_id']].copy()
author_career_2 = author_career_2.loc[author_career_2['author_id'].isin(set(author_stats['author_id']))]
author_career_2 = author_career_2.merge(pub2ids[['oaid', 'magid']], on='oaid', how='left')
author_career_2 = author_career_2.merge(pub2year[['PaperID', 'Year']], left_on='magid', right_on='PaperID', how='inner')
author_career_2 = (
    author_career_2.groupby('author_id', as_index=False)['Year'].min().rename(columns={'Year': 'first_entry_year'})
)

author_panel = author_stats.merge(author_career_1, on='author_id', how='left').merge(
    author_career_2, on='author_id', how='left'
)
author_panel.to_csv(OUT_DIR / 'author_panel.csv', index=False)

author_panel.groupby('treated', dropna=False)[
    ['pub_count', 'first_independent_year', 'first_entry_year', 'mean_team_size']
].mean()